# 🧬 장기 실종아동 나이 변환(Age Progression) 시스템
## AI Hub 4대 한국인 데이터셋 + 5TB 구글 드라이브 + Colab T4 GPU 올인원 파이프라인

본 노트북은 **내 컴퓨터의 용량(SSD/HDD)을 1MB도 사용하지 않고**, Google Colab의 초고속 인터넷망과 T4 GPU(16GB VRAM), 그리고 **5TB 구글 드라이브**를 직접 연결하여 AI Hub의 4대 한국인 특화 데이터셋을 다운로드 및 전처리하고 대규모 학습까지 완료하는 통합 파이프라인입니다.

---
### 📚 학습에 활용되는 AI Hub 4대 핵심 데이터셋 (자동승인 번호 확인 완료):
1. **`528`**: **가족 관계가 알려진 얼굴 이미지 데이터** (Kinship Cross-Attention $L_{Kinship}$ 유전 융합)
2. **`71415`**: **안면 인식 에이징(aging) 이미지 데이터** (한국인 생애주기별 노화 곡선 $L_{Age}$ 학습)
3. **`71785`**: **안면 랜드마크 데이터** (정밀 안면 정렬 및 하관 골격 마진 크롭)
4. **`85`**: **한국인 헤어스타일 이미지** (성인기 연령/성별 헤어 질감 및 다양성 학습)

---
### 📌 런타임 유형 설정 확인:
상단 메뉴 **[런타임] ➔ [런타임 유형 변경] ➔ 하드웨어 가속기: [T4 GPU]** 로 설정되어 있는지 확인하세요.

In [ ]:
# [Step 1] GPU 가속기 (T4 16GB VRAM) 상태 확인
!nvidia-smi

In [ ]:
# [Step 2] 5TB Google Drive 마운트 (내 컴퓨터 용량 소모 0MB)
from google.colab import drive
drive.mount('/content/drive')

# 구글 드라이브 내 데이터 보관용 폴더 생성
!mkdir -p /content/drive/MyDrive/aihub_data/raw
!mkdir -p /content/drive/MyDrive/aihub_data/checkpoints
print("[OK] Google Drive connected successfully!")

In [ ]:
# [Step 3] GitHub 저장소 클론 및 최신 코드 동기화
%cd /content
!rm -rf kinship-age-progression
!git clone https://github.com/ggg2214/kinship-age-progression.git
%cd kinship-age-progression
!ls -la

In [ ]:
# [Step 4] 필수 패키지 설치
!pip install -q torch torchvision fastapi uvicorn python-multipart Pillow

In [ ]:
# [Step 5] AI Hub 공식 다운로더 (aihubshell) 설치 및 권한 부여
!curl -s -o aihubshell https://api.aihub.or.kr/api/aihubshell.do
!chmod +x aihubshell
!sudo cp aihubshell /usr/bin/
!aihubshell --help || true
print("[OK] aihubshell installed successfully!")

In [ ]:
# [Step 6] AI Hub API Key 설정
import os

# Infisical에 등록된 사용자의 공식 AI Hub API Key
AI_HUB_API_KEY = "E610047C-F6E0-45BB-B10D-7D72BD55B6DA"
os.environ["AI_HUB_API_KEY"] = AI_HUB_API_KEY
print(f"[OK] AI Hub API Key configured: {AI_HUB_API_KEY[:8]}********")

In [ ]:
# [Step 7] 승인 완료된 4대 데이터셋 Google Drive로 직통 다운로드
# ✅ 확인 완료된 실제 데이터셋 번호(dataSetSn):
# - 528   : 가족 관계가 알려진 얼굴 이미지 데이터 (Kinship 핵심)
# - 71415 : 안면 인식 에이징(aging) 이미지 데이터 (노화 곡선 핵심)
# - 71785 : 안면 랜드마크 데이터
# - 85    : 한국인 헤어스타일 이미지

# 우선 나이 변환에 가장 핵심인 528(가족 관계)과 71415(에이징)를 다운로드합니다.
# (원하시면 아래 리스트에 "71785", "85"를 추가하여 4개 전체를 다운로드하실 수 있습니다)
DATASET_KEYS = ["528", "71415"]

RAW_DIR = "/content/drive/MyDrive/aihub_data/raw"
for dkey in DATASET_KEYS:
    print(f"\n[INFO] Starting direct download for dataset {dkey} into Google Drive...")
    !python age_progression/data/aihub_pipeline.py \
        --action download \
        --apikey $AI_HUB_API_KEY \
        --datasetkey {dkey} \
        --raw-dir {RAW_DIR}

# 다운로드 결과 용량 및 파일 목록 정밀 점검
print("\n=== [다운로드 결과 점검] ===")
!ls -lh /content/drive/MyDrive/aihub_data/raw || true
!find /content/drive/MyDrive/aihub_data/raw -type f | wc -l | xargs -I{} echo "총 다운로드 파일 수: {} 개"

In [ ]:
# [Step 8] 초경량 256x256 얼굴 크롭, 정렬 및 가족 Triplet 매니페스트 자동 생성
# 수백 GB의 불필요한 배경을 제거하고 핵심 얼굴만 256x256으로 정제하여 5~10GB로 압축 다이어트

RAW_DIR = "/content/drive/MyDrive/aihub_data/raw"
ALIGNED_DIR = "/content/data_aihub_aligned"
MANIFEST_PATH = "/content/data_aihub_aligned/manifest_aihub.json"

!python age_progression/data/aihub_pipeline.py \
    --action preprocess \
    --raw-dir {RAW_DIR} \
    --aligned-dir {ALIGNED_DIR} \
    --manifest {MANIFEST_PATH}

# 정제된 256x256 데이터셋을 단일 압축본으로 구글 드라이브에 안전 백업 (Colab 초고속 로딩용)
!python age_progression/data/aihub_pipeline.py \
    --action package \
    --aligned-dir {ALIGNED_DIR} \
    --archive /content/drive/MyDrive/aihub_data/korean_face_aligned_256.tar.gz

print("[OK] 256x256 Aligned Dataset & Triplet Manifest ready!")

In [ ]:
# [Step 9] 3단계 점진적 커리큘럼 학습 (Curriculum Learning) 가동
# Phase 1 (Reconstruction & ID) -> Phase 2 (한국인 노화 특화) -> Phase 3 (혈연 Cross-Attention 융합)

MANIFEST_PATH = "/content/data_aihub_aligned/manifest_aihub.json"

# 만약 AI Hub 다운로드 진행 전이거나 파일이 없을 경우 FG-NET 실제 종단면 벤치마크 데이터로 자동 대체
if not os.path.exists(MANIFEST_PATH) or os.path.getsize(MANIFEST_PATH) < 100:
    print("[INFO] AI Hub manifest not found or empty. Using real longitudinal FG-NET benchmark dataset...")
    !python age_progression/data/download_fgnet.py
    MANIFEST_PATH = "data_fgnet/manifest_fgnet.json"

!python age_progression/train_curriculum.py \
    --device cuda \
    --image-size 256 \
    --batch-size 4 \
    --epochs 5 \
    --manifest {MANIFEST_PATH} \
    --save-path /content/drive/MyDrive/aihub_data/checkpoints/age_progression_korean_best.pt

print("[SUCCESS] 3-Phase Curriculum Training Completed!")

In [ ]:
# [Step 10] 학습 완료된 한국인 모델로 성장 타임랩스(10~60세) 시각화 검증
CKPT_PATH = "/content/drive/MyDrive/aihub_data/checkpoints/age_progression_korean_best.pt"
if not os.path.exists(CKPT_PATH):
    CKPT_PATH = "checkpoints/age_progression_curriculum.pt"

!python age_progression/visualize_samples.py \
    --child public/assets/real_korean_child_06.jpg \
    --father public/assets/real_korean_father_38.jpg \
    --mother public/assets/real_korean_mother_36.jpg \
    --ages 10 20 30 40 50 60 \
    --source-age 6 \
    --checkpoint {CKPT_PATH} \
    --output outputs/korean_aihub_timelapse.png

from IPython.display import Image, display
display(Image('outputs/korean_aihub_timelapse.png'))

In [ ]:
# [Step 11] (중요) 대용량 원본 파일 정리 (Clean-Up)
# 질문: "AI 학습을 시키고 난 뒤 다운받은 파일을 삭제해도 되는가?"
# 답변: 네! 학습이 끝나면 수십~수백 GB의 날것 원본 파일은 완전히 삭제해도 모델 실행에 아무 문제 없습니다.
#      최종 모델 가중치(.pt)와 경량화된 256x256 압축본만 구글 드라이브에 안전하게 보관됩니다.

# 원본 대용량 다운로드 폴더 삭제를 원하시면 아래 코드의 주석(#)을 풀고 실행하세요:
# !rm -rf /content/drive/MyDrive/aihub_data/raw
# print("[OK] Raw bulky files safely cleaned up! Storage freed!")

# 최종 보관된 결과물 용량 확인
!ls -lh /content/drive/MyDrive/aihub_data/checkpoints/
!ls -lh /content/drive/MyDrive/aihub_data/korean_face_aligned_256.tar.gz || true